In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import bbknn
import scrublet as scr
import matplotlib.pyplot as plt
import scanpy.external as sce

In [ ]:
# data dir
input_dir = " "
output_dir = " "
nohub_gene_dir = " "
os.listdir(input_dir)

In [ ]:
## pre work
file_list = ["ACC","ANS","brain_healthy","BRCA","CESC","CRC","ESCC","GC","HNSC","KIRC","liver_healthy","LSCC","LUAD","LUSC","lymph_node_healthy",
             "MA","NPC","NSCLC","PAAD","PNET","SARC","TGCT","THCA"]
# hub_cell_type = ["T&NK_cell","Epithelia","Fibroblast","Endothelia","B_cell","Plasma_cell","Myeloid_cell","Mast_cell","Neutrophils"]
# hub_cell_type = ["Epithelia_tumor","Endothelia","Fibroblast","B_cell","Plasma_cell","Myeloid_cell","Neutrophils","Epithelia","T&NK_cell"]
hub_cell_type = ["T_cell"]
batch_method = "bbknn"
filter_nohub_gene = False
log_data_copy = True
bbknn_ridge = False
regress = True
random_state = 123
resolution = 1

nohub_gene = pd.read_excel(nohub_gene_dir,sheet_name="gene_name")

In [ ]:

# data run
for celltype in hub_cell_type:
    print(f"################################## {celltype}  process ##################################")
    
    scRNA_current = []

    ## diff data read
    for file in file_list:

        if os.path.isfile(f"{input_dir}/{file}/scRNA_{celltype}.h5"):
            print(f"      ----------------- {file}  read")

            ### data read
            celltype_current = diopy.input.read_h5(file = f"{input_dir}/{file}/scRNA_{celltype}.h5")

            ### data add
            scRNA_current.append(celltype_current)
    
            del celltype_current
            gc.collect()

    ## dir create
    output_file = f"{output_dir}/{celltype}"
    os.makedirs(output_file, exist_ok=True)

    ## data merge
    scRNA_current = sc.concat(scRNA_current)
    scRNA_current.obs_names_make_unique()

    ## origin data save
    # scRNA_current.write_h5ad(f"{output_file}/scRNA_origin.h5ad", compression="gzip")

    ## filter useful gene
    if filter_nohub_gene == True:
        filter_gene = scRNA_current.var_names.isin(list(nohub_gene.gene_name))
        keep_gene = np.invert(filter_gene)
        scRNA_current = scRNA_current[:,keep_gene]

    ## filter gene data save
    scRNA_current.write_h5ad(f"{output_file}/scRNA_gene_filter.h5ad", compression="gzip")
    diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_gene_filter.h5",save_X=False)

    ## data nor
    if log_data_copy == True:
        scRNA_current.layers["counts"] = scRNA_current.X.copy()
        sc.pp.normalize_total(scRNA_current, target_sum=1e4)
        sc.pp.log1p(scRNA_current)
        scRNA_current.layers["log1p"] = scRNA_current.X.copy()
        scRNA_current.raw = scRNA_current
    else:
        sc.pp.normalize_total(scRNA_current, target_sum=1e4)
        sc.pp.log1p(scRNA_current)
        scRNA_current.raw = scRNA_current

    ## HVG
    if celltype == "Epithelia_tumor":
        sc.pp.highly_variable_genes(scRNA_current, flavor='seurat',batch_key= "tumor_code", n_top_genes=5000)
    else: 
        sc.pp.highly_variable_genes(scRNA_current, flavor='seurat',batch_key= "tumor_code", n_top_genes=3000)
    scRNA_current = scRNA_current[:, scRNA_current.var.highly_variable]
    gc.collect()

    ## regress out effects of total counts per cell and the percentage of mitochondrial genes expressed
    if regress == True:
        sc.pp.regress_out(scRNA_current, ['nCount_RNA','percent.mt'])
    gc.collect()

    ## data scale
    sc.pp.scale(scRNA_current, max_value=10)

    ## run PCA
    sc.tl.pca(scRNA_current, svd_solver='arpack', use_highly_variable=True, n_comps=50)
    # sc.pl.pca_variance_ratio(scRNA_current, n_pcs=50, log=True)

    ## pca data save
    # scRNA_current.write_h5ad(f"{output_file}/scRNA_pca.h5ad", compression="gzip")

    ## tumor cell no bacth plot
    if celltype == "Epithelia_tumor":
        scRNA_copy = scRNA_current.copy()     
        sc.pp.neighbors(scRNA_copy)
        sc.tl.leiden(scRNA_copy, resolution=resolution, n_iterations=-1, random_state=random_state)
        sc.tl.umap(scRNA_copy)
        sc.pl.umap(scRNA_copy, color='study_ID',show = False)
        plt.savefig(f'{output_file}/scRNA_tumor_cell_in_batch_umap_study.png', dpi=3000)
        sc.pl.umap(scRNA_copy, color='leiden',show = False)
        plt.savefig(f'{output_file}/scRNA_tumor_cell_in_batch_umap_cluster.png', dpi=3000)
        sc.pl.umap(scRNA_copy, color='tumor_code',show = False)
        plt.savefig(f'{output_file}/scRNA_tumor_cell_in_batch_umap_tumor_code.png', dpi=3000)
        scRNA_copy.write_h5ad(f"{output_file}/scRNA_tumor_cell_in_batch.h5ad", compression="gzip")
        del scRNA_copy


    ## remove batch
    sc.external.pp.bbknn(scRNA_current, batch_key= "sample_ID", n_pcs = 50)
    # bbknn.bbknn(scRNA_current, batch_key= "sample_ID", n_pcs = 50)

    ## data save
    diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_bbknn.h5",save_X=False)
    scRNA_current.write_h5ad(f"{output_file}/scRNA_bbknn.h5ad", compression="gzip")


    ## space free
    del scRNA_current
    gc.collect()



In [ ]:
celltype = "Epithelia_tumor"
scRNA_current = []

## diff data read
for file in file_list:

    if os.path.isfile(f"{input_dir}/{file}/scRNA_{celltype}.h5"):
        print(f"      ----------------- {file}  read")

        ### data read
        celltype_current = diopy.input.read_h5(file = f"{input_dir}/{file}/scRNA_{celltype}.h5")

        ### data add
        scRNA_current.append(celltype_current)
    
        del celltype_current
        gc.collect()

## data merge
scRNA_current = sc.concat(scRNA_current)
scRNA_current.obs_names_make_unique()

scRNA_current.layers["counts"] = scRNA_current.X.copy()
sc.pp.normalize_total(scRNA_current, target_sum=1e4)
sc.pp.log1p(scRNA_current)
scRNA_current.layers["log1p"] = scRNA_current.X.copy()
scRNA_current.raw = scRNA_current

sc.pp.highly_variable_genes(scRNA_current, flavor='seurat',batch_key= "tumor_code", n_top_genes=25000)

In [ ]:
a

In [ ]:
a = scRNA_current.var["highly_variable"]
a.to_csv(f" ")

In [ ]:
scRNA_current.write_h5ad(f"{output_file}/scRNA_bbknn.h5ad", compression="gzip")

In [ ]:
scRNA_current

In [ ]:
output_file